# Fabric Anomaly Detection with AnomalyDINO (DINOv2 + Memory Bank)

This notebook implements the pipeline you described, built for **Kaggle**:

```
Input Image -> DINOv2 (feature extraction) -> Feature Representation
   -> compare Test Features vs Normal Feature Memory (kNN similarity search)
   -> Anomaly Score Map -> Threshold + Post-processing -> Defect Localization
```

**How it works, in plain terms:**
1. We only need *normal* (defect-free) fabric images to "train" this model — there is no
   backpropagation/weight training involved. DINOv2 is a frozen, pretrained vision
   transformer; we just use it as a feature extractor.
2. Every normal training image is converted into a grid of small patch-level feature
   vectors. All of these vectors together form the **memory bank** ("Normal Feature
   Memory" in your diagram).
3. For a new (test) image, we extract the same kind of patch features, and for every
   patch we ask: "what's the closest thing to this in the memory bank of normal
   patches?" A large distance means that patch looks unlike anything normal — i.e. a
   likely defect.
4. Those per-patch distances are reshaped into a 2D **anomaly map**, upsampled to the
   original image size, thresholded, and used to localize/flag defects.

## Expected dataset layout on Kaggle

This notebook assumes an MVTec-AD-style folder layout, which is the most common
convention for anomaly detection datasets:

```
DATASET_ROOT/
├── train/
│   └── good/                     <- only normal fabric images
├── test/
│   ├── good/                     <- normal test images
│   ├── hole/                     <- one folder per defect type
│   ├── stain/
│   └── ...
└── ground_truth/                 <- OPTIONAL, only needed for pixel-level scoring
    ├── hole/
    ├── stain/
    └── ...
```

If your dataset is organized differently (e.g. everything in one folder with a CSV of
labels, or just `good/` vs `defect/`), skip to the **"Adapting to a different folder
layout"** note near the end — the core DINOv2 + memory bank logic doesn't change, only
the file-listing code does.

Update `CONFIG["DATASET_ROOT"]` in the next section to point at your dataset under
`/kaggle/input/...` (visible in the right-hand "Data" panel of your Kaggle notebook).


### A note on your 64x64 images

DINOv2's image processor always resizes whatever you feed it to `CONFIG["IMAGE_SIZE"]`,
so 64x64 inputs work fine as-is — no manual resizing needed, and nothing below has to
change to handle them. Two practical consequences worth knowing:

- **Upsampling is happening.** 64x64 -> 224x224 is ~3.5x upsampling per side. It's
  interpolated (blurry) beyond the original 64x64 detail, but DINOv2 still extracts
  meaningful texture/structure features from it — this is normal and works well in
  practice for defect detection.
- **Localization precision is capped by your source resolution.** A defect that's a
  couple of pixels wide in the original 64x64 image will look bigger for `IMAGE_SIZE=224`,
  but you're not getting extra real detail — the anomaly *map* will be smooth/coarse. If
  precise defect boundaries matter, consider whether higher-resolution source images are
  available; if 64x64 is genuinely all you have, this pipeline still works well for
  **detecting** (this image/region is anomalous) even if pixel-perfect **boundaries** are
  less meaningful.
- If you want to push resolution higher anyway (e.g. `IMAGE_SIZE=336` or `518`) to get a
  finer-grained anomaly map for visualization, you can — just know you're not adding real
  information, only smoother interpolation.


## 0. Install & import dependencies

In [ ]:
# -q = quiet install. These are the only non-default packages we need:
# transformers -> gives us the pretrained DINOv2 model + its image preprocessor
# scikit-learn -> nearest-neighbour search + AUROC evaluation metric
# opencv-python-headless -> resizing/smoothing the anomaly map (no GUI needed on Kaggle)
!pip install -q transformers scikit-learn opencv-python-headless


In [ ]:
import os                                  # filesystem paths
from glob import glob                      # pattern-based file listing
import random                              # for picking random visualization samples

import numpy as np                         # numerical arrays / linear algebra
import torch                               # deep learning tensors + GPU support
import cv2                                 # image resizing & smoothing
import matplotlib.pyplot as plt            # plotting anomaly maps
from PIL import Image                      # loading image files
from tqdm import tqdm                      # progress bars

from transformers import AutoImageProcessor, AutoModel   # pretrained DINOv2
from sklearn.neighbors import NearestNeighbors            # kNN similarity search
from sklearn.metrics import roc_auc_score                 # evaluation metric

print("Torch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())


## 1. Configuration

Edit `DATASET_ROOT` (and anything else you like) before running.

In [ ]:
CONFIG = {
    # Root folder of your Kaggle dataset. CHANGE THIS to your dataset's path,
    # e.g. "/kaggle/input/fabric-defect-dataset".
    "DATASET_ROOT": "/kaggle/input/your-fabric-dataset",

    # Pretrained DINOv2 backbone. Bigger = more accurate but slower/more VRAM.
    # Options: facebook/dinov2-small, -base, -large, -giant
    "MODEL_NAME": "facebook/dinov2-base",

    # Images are resized to this square resolution before DINOv2 sees them.
    # Must be divisible by 14 (DINOv2's patch size). 224 = 16 * 14.
    # NOTE: your source images are 64x64. DINOv2's processor will upsample them
    # to whatever IMAGE_SIZE you set here regardless of their original size —
    # 224 is a good default: it's DINOv2's native training resolution, gives a
    # 16x16 = 256 patch grid (plenty for a 64x64-derived image), and avoids the
    # extra blur/compute of upsampling all the way to 518 for no real benefit
    # (you have no extra real detail past 64x64 to recover anyway).
    "IMAGE_SIZE": 224,

    # How many nearest memory-bank neighbours to average per test patch.
    "K_NEIGHBORS": 1,

    # Cap on how many patch vectors the memory bank keeps (coreset subsampling).
    # Set to None to keep every patch from every training image (uses more RAM/time).
    "MAX_MEMORY_SIZE": 20000,

    # Percentile of NORMAL training scores used as the good/defective threshold.
    # 99 means ~1% of normal training images would be (mis)flagged as defective.
    "THRESHOLD_PERCENTILE": 99,

    # Run on GPU if Kaggle gives you one (Settings -> Accelerator -> GPU), else CPU.
    "DEVICE": "cuda" if torch.cuda.is_available() else "cpu",
}

print(CONFIG)


## 2. DINOv2 feature extractor

This class wraps the pretrained DINOv2 model and turns one image into a **grid of
patch-level feature vectors** — this is the "DINOv2 Vision Transformer -> Feature
Representation" stage of your diagram.

In [ ]:
class DINOv2FeatureExtractor:
    """Extracts a spatial grid of DINOv2 patch embeddings from an image."""

    def __init__(self, model_name, image_size, device):
        # The processor resizes/normalizes images exactly the way DINOv2 was trained on.
        self.processor = AutoImageProcessor.from_pretrained(model_name)
        # Force a fixed square input size so every image yields the same number
        # of patch tokens (needed so they line up on one consistent grid).
        self.processor.size = {"height": image_size, "width": image_size}
        self.processor.crop_size = {"height": image_size, "width": image_size}

        self.model = AutoModel.from_pretrained(model_name)  # load pretrained weights
        self.model.eval()                                   # inference mode (no dropout)
        self.model.to(device)                                # move weights to GPU/CPU

        self.device = device
        self.image_size = image_size
        # DINOv2 splits the image into non-overlapping 14x14 pixel patches,
        # so the token grid's side length is image_size / 14.
        self.grid_size = image_size // 14

    @torch.no_grad()  # disables gradient tracking -> faster, less memory
    def extract_patch_features(self, pil_image):
        """Returns an array of shape (grid_size*grid_size, C): one L2-normalized
        feature vector per image patch, in row-major (top-left to bottom-right) order."""

        # PIL image -> normalized tensor batch, shape (1, 3, image_size, image_size).
        inputs = self.processor(images=pil_image, return_tensors="pt")
        inputs = {k: v.to(self.device) for k, v in inputs.items()}

        outputs = self.model(**inputs)                 # run DINOv2 forward pass
        last_hidden_state = outputs.last_hidden_state   # (1, 1+num_patches, C)

        # Token 0 is the global [CLS] token; we only want the per-patch tokens.
        patch_tokens = last_hidden_state[:, 1:, :]      # (1, num_patches, C)
        patch_tokens = patch_tokens.squeeze(0)          # (num_patches, C)

        # L2-normalize so Euclidean distance between vectors behaves like
        # (1 - cosine similarity) — a standard trick for feature comparison.
        patch_tokens = torch.nn.functional.normalize(patch_tokens, dim=1)

        return patch_tokens.cpu().numpy()               # back to plain numpy array


## 3. Dataset utilities

Simple helpers to list image files and load them as RGB.

In [ ]:
def list_images(folder):
    """Return a sorted list of image paths inside `folder`."""
    exts = (".png", ".jpg", ".jpeg", ".bmp", ".tif", ".tiff")
    paths = []
    for ext in exts:
        paths.extend(glob(os.path.join(folder, f"*{ext}")))
        paths.extend(glob(os.path.join(folder, f"*{ext.upper()}")))
    return sorted(set(paths))


def load_image(path):
    """Load an image file and force 3-channel RGB (DINOv2 expects RGB)."""
    return Image.open(path).convert("RGB")


## 4. Build the normal feature memory bank

This runs every *normal* training image through DINOv2 and stacks all of their patch
vectors together. This matrix is the "Normal Feature Memory (Reference Database)" box
in your diagram.

Because a full memory bank can get huge (num_images x num_patches vectors), we also
provide **coreset subsampling**: a greedy "farthest point" selection that keeps a
smaller, representative subset — the standard trick used in PatchCore/AnomalyDINO-style
methods to keep nearest-neighbour search fast without losing coverage of normal
appearance variation.

In [ ]:
def build_memory_bank(extractor, good_image_paths):
    """Extracts and stacks patch features from every normal training image."""
    all_features = []
    for path in tqdm(good_image_paths, desc="Extracting normal features"):
        img = load_image(path)
        feats = extractor.extract_patch_features(img)   # (num_patches, C)
        all_features.append(feats)
    # Stack every image's patches into one big (total_patches, C) matrix.
    return np.concatenate(all_features, axis=0)


def coreset_subsample(memory_bank, max_size, seed=0):
    """Greedy farthest-point subsampling down to `max_size` vectors.
    Keeps the memory bank small/fast while preserving its overall shape
    (diverse normal patterns), instead of just randomly dropping points."""
    if max_size is None or len(memory_bank) <= max_size:
        return memory_bank

    rng = np.random.default_rng(seed)
    n = len(memory_bank)

    chosen_idx = [int(rng.integers(0, n))]               # start from one random point
    # For every point, track its distance to the CLOSEST point chosen so far.
    min_dists = np.linalg.norm(memory_bank - memory_bank[chosen_idx[0]], axis=1)

    for _ in tqdm(range(max_size - 1), desc="Coreset subsampling"):
        next_idx = int(np.argmax(min_dists))              # farthest-from-chosen point
        chosen_idx.append(next_idx)
        new_dists = np.linalg.norm(memory_bank - memory_bank[next_idx], axis=1)
        min_dists = np.minimum(min_dists, new_dists)       # update running minimum

    return memory_bank[chosen_idx]


## 5. Similarity search & anomaly map

For a test image's patches, we find each patch's distance to its nearest neighbour in
the memory bank, reshape those distances into a 2D map, and upsample it back to full
image resolution. This is the "Feature Similarity Search -> Anomaly Score Map" stage.

In [ ]:
def build_nn_index(memory_bank, k):
    """Fits a k-NN index over the memory bank for fast distance lookups."""
    nn_index = NearestNeighbors(n_neighbors=k, metric="euclidean", n_jobs=-1)
    nn_index.fit(memory_bank)
    return nn_index


def compute_anomaly_map(patch_features, nn_index, grid_size, image_size):
    """Turns per-image patch features into a full-resolution anomaly heatmap
    and a single scalar image-level anomaly score."""

    distances, _ = nn_index.kneighbors(patch_features)     # (num_patches, k)
    patch_scores = distances.mean(axis=1)                  # avg over k neighbours

    # Put the flat list of per-patch scores back into their 2D spatial layout.
    score_grid = patch_scores.reshape(grid_size, grid_size)

    # Upsample the coarse patch-grid map to full image resolution (smooth interpolation).
    anomaly_map = cv2.resize(
        score_grid.astype(np.float32), (image_size, image_size),
        interpolation=cv2.INTER_CUBIC
    )
    # Light Gaussian blur to reduce blocky/noisy artifacts from upsampling.
    anomaly_map = cv2.GaussianBlur(anomaly_map, (5, 5), sigmaX=4)

    # A single per-image score: the single most anomalous point in the map.
    image_score = float(anomaly_map.max())

    return anomaly_map, image_score


## 6. Thresholding & post-processing

In [ ]:
def compute_threshold(normal_scores, percentile):
    """Sets the good/defective cutoff as a high percentile of scores measured
    on NORMAL training images, so most normal images score below it."""
    return float(np.percentile(normal_scores, percentile))


def get_binary_mask(anomaly_map, threshold):
    """Pixels scoring above the threshold are flagged as defective (1)."""
    return (anomaly_map > threshold).astype(np.uint8)


## 7. Visualization helper

In [ ]:
def visualize_result(pil_image, anomaly_map, mask, title=""):
    fig, axes = plt.subplots(1, 3, figsize=(12, 4))

    axes[0].imshow(pil_image)
    axes[0].set_title("Input")
    axes[0].axis("off")

    axes[1].imshow(pil_image)
    axes[1].imshow(anomaly_map, cmap="jet", alpha=0.5)   # heatmap overlaid on the image
    axes[1].set_title("Anomaly Heatmap")
    axes[1].axis("off")

    axes[2].imshow(mask, cmap="gray")
    axes[2].set_title("Defect Mask")
    axes[2].axis("off")

    fig.suptitle(title)
    plt.tight_layout()
    plt.show()


## 8. Build the memory bank ("training")

There's no gradient-based training here — this cell just runs every normal image
through DINOv2 once and stores the resulting features on disk, so you don't have to
redo this expensive step if the Kaggle kernel restarts.

In [ ]:
extractor = DINOv2FeatureExtractor(
    model_name=CONFIG["MODEL_NAME"],
    image_size=CONFIG["IMAGE_SIZE"],
    device=CONFIG["DEVICE"],
)

train_good_dir = os.path.join(CONFIG["DATASET_ROOT"], "train", "good")
good_paths = list_images(train_good_dir)
print(f"Found {len(good_paths)} normal training images")
assert len(good_paths) > 0, "No training images found — check CONFIG['DATASET_ROOT']"

raw_memory_bank = build_memory_bank(extractor, good_paths)
print("Raw memory bank shape:", raw_memory_bank.shape)

memory_bank = coreset_subsample(raw_memory_bank, CONFIG["MAX_MEMORY_SIZE"])
print("Final memory bank shape:", memory_bank.shape)

# /kaggle/working/ is the writable output directory on Kaggle.
np.save("/kaggle/working/memory_bank.npy", memory_bank)

nn_index = build_nn_index(memory_bank, CONFIG["K_NEIGHBORS"])


## 9. Calibrate the decision threshold

We score the normal training images themselves (they should mostly score low) and use
a high percentile of those scores as the good/defective cutoff.

In [ ]:
train_scores = []
for path in tqdm(good_paths, desc="Scoring normal training images"):
    img = load_image(path)
    feats = extractor.extract_patch_features(img)
    _, score = compute_anomaly_map(feats, nn_index, extractor.grid_size, CONFIG["IMAGE_SIZE"])
    train_scores.append(score)

threshold = compute_threshold(train_scores, CONFIG["THRESHOLD_PERCENTILE"])
print("Decision threshold:", threshold)


## 10. Run inference on the test set + evaluate

Every subfolder inside `test/` is treated as a category: `good` = normal (label 0),
anything else = a defect type (label 1). We compute an anomaly map + score for every
test image and, if labels are available, report image-level AUROC.

In [ ]:
test_root = os.path.join(CONFIG["DATASET_ROOT"], "test")
categories = sorted(os.listdir(test_root))

results = []  # each entry: path, category, label, score, anomaly_map

for category in categories:
    cat_dir = os.path.join(test_root, category)
    if not os.path.isdir(cat_dir):
        continue
    label = 0 if category == "good" else 1   # 0 = normal, 1 = defective

    for path in list_images(cat_dir):
        img = load_image(path)
        feats = extractor.extract_patch_features(img)
        amap, score = compute_anomaly_map(
            feats, nn_index, extractor.grid_size, CONFIG["IMAGE_SIZE"]
        )
        results.append({
            "path": path, "category": category, "label": label,
            "score": score, "anomaly_map": amap,
        })

print(f"Scored {len(results)} test images across categories: {categories}")

y_true = [r["label"] for r in results]
y_score = [r["score"] for r in results]

# AUROC only makes sense if we have both classes present.
if len(set(y_true)) > 1:
    image_auroc = roc_auc_score(y_true, y_score)
    print(f"Image-level AUROC: {image_auroc:.4f}")
else:
    print("Only one class present in test set — skipping AUROC.")


## 11. (Optional) Pixel-level evaluation

Only runs if a `ground_truth/` folder with per-defect binary masks exists. Skips
gracefully otherwise.

In [ ]:
gt_root = os.path.join(CONFIG["DATASET_ROOT"], "ground_truth")

if os.path.isdir(gt_root):
    pixel_true, pixel_score = [], []

    for r in results:
        if r["category"] == "good":
            # Normal images -> mask is all zeros (no defect pixels anywhere).
            mask = np.zeros((CONFIG["IMAGE_SIZE"], CONFIG["IMAGE_SIZE"]), dtype=np.uint8)
        else:
            fname = os.path.splitext(os.path.basename(r["path"]))[0]
            candidates = glob(os.path.join(gt_root, r["category"], f"{fname}*"))
            if not candidates:
                continue  # no matching mask file found, skip this image
            gt_img = Image.open(candidates[0]).convert("L")
            gt_img = gt_img.resize((CONFIG["IMAGE_SIZE"], CONFIG["IMAGE_SIZE"]))
            mask = (np.array(gt_img) > 127).astype(np.uint8)

        pixel_true.append(mask.flatten())
        pixel_score.append(r["anomaly_map"].flatten())

    pixel_true = np.concatenate(pixel_true)
    pixel_score = np.concatenate(pixel_score)
    pixel_auroc = roc_auc_score(pixel_true, pixel_score)
    print(f"Pixel-level AUROC: {pixel_auroc:.4f}")
else:
    print("No ground_truth/ folder found — skipping pixel-level evaluation.")


## 12. Visualize a few sample results

In [ ]:
sample_results = random.sample(results, min(5, len(results)))

for r in sample_results:
    img = load_image(r["path"])
    mask = get_binary_mask(r["anomaly_map"], threshold)
    verdict = "DEFECTIVE" if r["score"] > threshold else "NORMAL"
    visualize_result(
        img, r["anomaly_map"], mask,
        title=f"{r['category']} | score={r['score']:.3f} | predicted: {verdict}"
    )


## Notes & tuning tips

- **No backprop training happens here.** DINOv2 stays frozen; "training" = building the
  memory bank from normal images. This is normal for this family of methods
  (PatchCore/AnomalyDINO) and is why it works well even with few defect examples (you
  don't need any defective images at all to build the model).
- **Speed/accuracy trade-off:** `dinov2-small` is much faster than `dinov2-base`/`-large`
  if you're compute-limited on Kaggle; try it first to validate the pipeline, then scale up.
- **`MAX_MEMORY_SIZE`**: larger memory bank = better coverage of normal fabric texture
  variation, but slower nearest-neighbour search. Coreset subsampling keeps this in check.
- **`THRESHOLD_PERCENTILE`**: lower it (e.g. 95) to catch more subtle defects at the cost
  of more false positives on normal fabric; raise it (e.g. 99.5) to be more conservative.
- **Fabric-specific tip:** many woven fabrics are naturally textured/periodic, which can
  make the memory bank pick up on texture "noise" as anomalies. If you see too many false
  positives, try training on more normal images that capture the fabric's natural weave
  variation, or increase `K_NEIGHBORS` to smooth scores.

### Adapting to a different folder layout
If your dataset doesn't follow `train/good`, `test/<category>`, e.g. it's just two flat
folders `normal/` and `defective/` (no train/test split), you only need to change how
paths are listed:
```python
normal_paths = list_images(os.path.join(CONFIG["DATASET_ROOT"], "normal"))
defective_paths = list_images(os.path.join(CONFIG["DATASET_ROOT"], "defective"))
# Use e.g. 80% of normal_paths to build the memory bank, hold out the rest (plus all
# defective_paths) as your test set, following the same pattern as sections 8-12 above.
```
Everything downstream (feature extraction, memory bank, kNN scoring, thresholding,
evaluation, visualization) stays exactly the same.
